# Loading and preparing data 

In [17]:
import scvi
import scanpy as sc

import pandas as pd
import subprocess
import os

In [18]:
print(scvi.__version__)

0.20.3


In [19]:
sc.set_figure_params(figsize=(5, 5))

In [20]:
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [21]:
scvi.settings.num_threads=48

# Import AnnData

In [22]:
adata = sc.read_h5ad('data/object/reynolds_et_al_2021/qc.h5ad')

# Register model

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    batch_key='sample_id', 
    categorical_covariate_keys=['donor_id'], 
    continuous_covariate_keys=['pMT_RNA', 'msCC_diff_RNA']
)

# Creating and training a model 

In [ ]:
model = scvi.model.SCVI(
    adata, 
    n_latent=10, 
    n_hidden=128, 
    n_layers=2, 
    gene_likelihood='nb'
)

In [ ]:
model.train()

In [ ]:
model.save('data/object/reynolds_et_al_2021/scvi/model/', overwrite=True)

# Update anndata object with scvi results 

In [ ]:
adata.obsm['latent'] = model.get_latent_representation()

# Dimensional reduction on scvi latent space 

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, use_rep='latent')
sc.tl.leiden(adata, resolution=1, key_added='leiden')
sc.tl.umap(adata, min_dist=1)

## UMAP

In [ ]:
sc.pl.umap(adata, color=['leiden', 'donor_id', 'sample_id', 'cc_phase_class', 'pMT_RNA', 'pHB_RNA', 'pRP_RNA', 'nCount_RNA', 'nFeature_RNA'], frameon=False, ncols=5, wspace=0.5)

# Save results

In [ ]:
adata.write('data/object/reynolds_et_al_2021/scvi/adata.h5ad')
adata.write('data/object/reynolds_et_al_2021/pp.h5ad')

In [ ]:
pd.DataFrame(adata.obsm['X_umap'], index=adata.obs.index).to_csv('data/object/reynolds_et_al_2021/components/umap.csv')
pd.DataFrame(adata.obs, index=adata.obs.index).to_csv('data/object/reynolds_et_al_2021/components/meta.csv')